Splitting labels into two groups fake news and reliable.

In [1]:
import pandas as pd
import ast
from rich.progress import track
from sklearn.feature_extraction.text import CountVectorizer
from sklearn import linear_model
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import swifter
import numpy as np

In [2]:
all_columns = [
    'id', 'domain', 'type', 'url', 'content', 'scraped_at', 'inserted_at', 
    'updated_at', 'title', 'authors', 'keywords', 'meta_keywords', 
    'meta_description', 'tags', 'summary']
converters_dict = {col: ast.literal_eval for col in all_columns}

training_set = pd.read_csv('../Group_project/training_set.csv', converters = converters_dict)

In [3]:
# creating list of 10000 most common words from vocabulary found in part 1.
full_voc = pd.read_csv('../Group_project/vocabulary_995000.csv')
sub_voc = set((full_voc.iloc[:10000])['word'].values.tolist())
print(sub_voc)

{'ethan', 'extend', 'oust', 'ginsburg', 'uncommon', 'ms', 'bargain', 'dilig', 'reev', 'netanyahu', 'alley', 'medvedev', 'privileg', 'undocu', 'reduct', 'death', 'barton', 'kramer', 'barack', 'prescrib', 'walid', 'individu', 'bs', 'lent', 'sheldon', 'stroll', 'sizabl', 'plain', 'omit', 'persuas', 'kean', 'harsh', 'striker', 'ventur', 'currenc', 'downplay', 'parungo', 'pheasant', 'chad', 'cynthia', 'atlas', 'sold', 'herbicid', 'asham', 'kahn', 'airway', 'shill', 'influenc', 'armenia', 'municip', 'determin', 'comprehens', 'putnam', 'yogurt', 'recipi', 'apocalypt', 'warsaw', 'financi', 'invas', 'tourism', 'megan', 'phototh', 'coyot', 'karlin', 'trick', 'intoler', 'egyptian', 'iwb', 'bds', 'heavier', 'mural', 'collis', 'wrath', 'lionel', 'nutrient', 'stand', 'condit', 'standpoint', 'artifici', 'hpv', 'eaten', 'clint', 'thu', 'distributor', 'wive', 'load', 'psychiatrist', 'quillfeath', 'battalion', 'melbourn', 'derail', 'cultiv', 'share', 'headlin', 'williamson', 'antarct', 'lemon', 'notion'

In [58]:
types = {i[0] for i in training_set['type'] if len(i) < 2}
print(types)
fake_news = {'fake', 'satir', 'bias', 'conspiraci', 'junksci', 'hate', 'rumor'}
reliable_news = {'clickbait', 'unreli', 'polit', 'reliabl'}

fake_only_rel = {'clickbait', 'unreli', 'polit', 'fake', 'satir', 'bias', 'conspiraci', 'junksci', 'hate', 'rumor'}
rel_only_rel = {'reliabl'}

def filter_type(type, fake, rel):
    if type in fake:
        return 'fake'
    elif type in rel:
        return 'reliable'

{'hate', 'rumor', 'bias', 'clickbait', 'unreli', 'reliabl', 'conspiraci', 'unknown', 'junksci', 'polit', 'satir', 'fake'}


In [59]:
def filter_strings(str_list, **kwargs):
    filtered = [word for word in str_list if word in sub_voc]
    joined = ' '.join(filtered)
    return joined

only_content = pd.DataFrame({})
only_content['content'] = training_set['content'].swifter.progress_bar(False).apply(filter_strings)
only_content['y_t'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))
only_content['rel_only_rel'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_only_rel, rel_only_rel))
print(only_content)

relevant_columns = [
    'content', 'title', 'authors', 'keywords', 'meta_keywords', 
    'meta_description', 'tags', 'summary']

def filter_df(df):
    f_df = pd.DataFrame({})
    for i in track(relevant_columns, 'processing...'):
        f_df[i] = df[i].swifter.progress_bar(False).apply(filter_strings)
    return f_df

filtered = filter_df(training_set)

filtered['y_t'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))
filtered['rel_only_rel'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_only_rel, rel_only_rel))
# print(filtered)

Output()

                                                  content       y_t  \
0       articl googl ali alfoneh assist compil politic...  reliable   
1       texa stand religi belief alleg teacher forc st...       NaN   
2       black kos editor dr woman african american ame...  reliable   
3       amaz moment gorilla reunit human friend back r...       NaN   
4       miss time alien disc bitcoin blockchain search...      fake   
...                                                   ...       ...   
795995  roger tlb don talk hear news exist danger chil...      fake   
795996  le du du moham abdel sur son twitter action du...      fake   
795997  arkansa gov mike huckabe call christian pastor...  reliable   
795998  news north korea latest attempt world attent a...  reliable   
795999  pm est updat comput firm post loss compar resu...  reliable   

       rel_only_rel  
0              fake  
1               NaN  
2              fake  
3               NaN  
4              fake  
...            

In [60]:
x_oc = only_content[only_content['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
y_oc = x_oc['y_t']
y_rel_only_rel = x_oc['rel_only_rel']

x_t = filtered[filtered['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
y_t = x_t['y_t']
y_t_rel_only_rel = x_t['rel_only_rel']

vectorizer = CountVectorizer()

X_oc = vectorizer.fit_transform(x_oc['content'])
Y_oc = np.array(y_oc)
Y_rel_only_rel = np.array(y_rel_only_rel)

transformer_list = [(f'vec_{col}', CountVectorizer(), col) for col in relevant_columns]
transformer = ColumnTransformer(transformers=transformer_list, remainder='drop')

X_train = transformer.fit_transform(x_t)
y_train = np.array(y_t)
y_train_rel_only_rel = np.array(y_t_rel_only_rel)

In [61]:
logr = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr.fit(X_oc, Y_oc)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [62]:
logr_rel_only_rel = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr_rel_only_rel.fit(X_oc, Y_rel_only_rel)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [63]:
logr_2_3 = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr_2_3.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [64]:
logr_2_3_rel_only_rel = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr_2_3_rel_only_rel.fit(X_train, y_train_rel_only_rel)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [8]:
test_set = pd.read_csv('../Group_project/test_set.csv', converters = converters_dict)

In [65]:
oc_test = pd.DataFrame({})
oc_test['content'] = test_set['content'].swifter.progress_bar(False).apply(filter_strings)
oc_test['y_t'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))
oc_test['rel_only_rel'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_only_rel, rel_only_rel))

filtered_test = filter_df(test_set)

filtered_test['y_t'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))
filtered_test['y_t_rel_only_rel'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_only_rel, rel_only_rel))

Output()

In [66]:
oc_t = oc_test[oc_test['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
oc_y = oc_t['y_t']
oc_y_rel_only_rel = oc_t['rel_only_rel']

OC_test = vectorizer.transform(oc_t['content'])
oc_y_test = np.array(oc_y)
oc_Y_rel_only_rel = np.array(oc_y_rel_only_rel)

x_test = filtered_test[filtered_test['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
yt_t = x_test['y_t']
yt_t_rel_only_rel = x_test['y_t_rel_only_rel']

X_test = transformer.transform(x_test)
y_test = np.array(yt_t)
y_test_rel_only_rel = np.array(yt_t_rel_only_rel)

In [67]:
predictions = logr.predict(OC_test)

predictions_rel_only_rel = logr_rel_only_rel.predict(OC_test)

predictions_2_3 = logr_2_3.predict(X_test)

predictions_2_3_rel_only_rel = logr_2_3_rel_only_rel.predict(X_test)

In [68]:
print(f'The acuracy score for the model is: {accuracy_score(oc_y_test, predictions)}')
print('The confusion matrix for the model:')
print(confusion_matrix(oc_y_test, predictions))
print('The classification report of the logistic model is:')
print(classification_report(oc_y_test, predictions))

The acuracy score for the model is: 0.8493961218836565
The confusion matrix for the model:
[[37706  5035]
 [ 8557 38952]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.82      0.88      0.85     42741
    reliable       0.89      0.82      0.85     47509

    accuracy                           0.85     90250
   macro avg       0.85      0.85      0.85     90250
weighted avg       0.85      0.85      0.85     90250



In [69]:
print(f'The acuracy score for the model is: {accuracy_score(oc_Y_rel_only_rel, predictions)}')
print('The confusion matrix for the model:')
print(confusion_matrix(oc_Y_rel_only_rel, predictions))
print('The classification report of the logistic model is:')
print(classification_report(oc_Y_rel_only_rel, predictions))

The acuracy score for the model is: 0.7098393351800554
The confusion matrix for the model:
[[44320 24244]
 [ 1943 19743]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.96      0.65      0.77     68564
    reliable       0.45      0.91      0.60     21686

    accuracy                           0.71     90250
   macro avg       0.70      0.78      0.69     90250
weighted avg       0.84      0.71      0.73     90250



In [70]:
print(f'The acuracy score for the model is: {accuracy_score(y_test, predictions_2_3)}')
print('The confusion matrix for the model:')
print(confusion_matrix(y_test, predictions_2_3))
print('The classification report of the logistic model is:')
print(classification_report(y_test, predictions_2_3))

The acuracy score for the model is: 0.9594238227146814
The confusion matrix for the model:
[[41295  1446]
 [ 2216 45293]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.95      0.97      0.96     42741
    reliable       0.97      0.95      0.96     47509

    accuracy                           0.96     90250
   macro avg       0.96      0.96      0.96     90250
weighted avg       0.96      0.96      0.96     90250



In [71]:
print(f'The acuracy score for the model is: {accuracy_score(y_test_rel_only_rel, predictions_2_3)}')
print('The confusion matrix for the model:')
print(confusion_matrix(y_test_rel_only_rel, predictions_2_3))
print('The classification report of the logistic model is:')
print(classification_report(y_test_rel_only_rel, predictions_2_3))

The acuracy score for the model is: 0.7122548476454293
The confusion matrix for the model:
[[43053 25511]
 [  458 21228]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.99      0.63      0.77     68564
    reliable       0.45      0.98      0.62     21686

    accuracy                           0.71     90250
   macro avg       0.72      0.80      0.69     90250
weighted avg       0.86      0.71      0.73     90250

